<a href="https://colab.research.google.com/github/M7md-salah/Used-Car-Price-Model-MachineLearning-Project/blob/main/Egyptian_Used_Cars_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚗 Egyptian Used Car Price Prediction Using Machine Learning

**A Complete End-to-End Machine Learning Project**

---

| | |
|---|---|
| **Dataset** | Egyptian used car listings (August 2025) |
| **Task** | Supervised regression — predict asking price in EGP |
| **Best Model** | Tuned Random Forest (200 estimators) |
| **Full-Test MAE** | ~565,695 EGP (n = 4,891) |
| **Mainstream MAE** | ~121,259 EGP (price ≤ 5 M EGP, n = 4,837) |
| **Mainstream R²** | 0.8253 |

---

### Project Stages
| Stage | Description |
|-------|-------------|
| 1 | Dataset Inspection |
| 2 | Data Cleaning |
| 3 | Exploratory Data Analysis (EDA) |
| 4 | Feature Engineering & Preprocessing |
| 5 | Baseline Model Training & Evaluation |
| 6 | Model Improvement (Tuning) |
| 7 | Final Validation & Prediction Pipeline |
| 8 | Final Assembly (this notebook) |


## ⚙️ Stage 0 — Environment Setup

Install dependencies and configure paths.

In [ ]:
# Install required libraries (skip if already installed in your environment)
import subprocess, sys

required = [
    "pandas>=1.5.0",
    "numpy>=1.23.0",
    "matplotlib>=3.6.0",
    "seaborn>=0.12.0",
    "scikit-learn>=1.1.0",
    "joblib>=1.2.0",
    "category_encoders>=2.6.0",
]

for pkg in required:
    try:
        pkg_name = pkg.split(">=")[0].replace("-", "_")
        __import__(pkg_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All dependencies are available.")

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
import pickle
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# ── Plotting style ──────────────────────────────────────────
plt.style.use("seaborn-v0_8-whitegrid")
PALETTE = sns.color_palette("husl", 10)
sns.set_palette(PALETTE)
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "figure.facecolor": "white",
})

# ── Path helper (works locally and in Colab) ────────────────
def get_project_root():
    """Locate the project root folder regardless of where this notebook runs."""
    # Try common Colab mount points first
    for candidate in [
        "/content/drive/MyDrive/ORG_project",
        "/content/ORG_project",
        os.path.abspath(os.path.join(os.getcwd(), "..")),  # one level up from notebooks/
        os.getcwd(),
    ]:
        if os.path.isfile(os.path.join(candidate, "data", "processed", "used_cars_cleaned.csv")):
            return candidate
    raise FileNotFoundError(
        "Cannot locate project root. "
        "Upload the project folder to Colab and set ROOT manually below."
    )

ROOT = get_project_root()
print(f"Project root: {ROOT}")

def path(*parts):
    return os.path.join(ROOT, *parts)

---
## 📋 Stage 1 — Dataset Inspection

Load the raw dataset and understand its structure before any cleaning.

In [ ]:
# Load raw dataset (read-only — never modify this file)
raw = pd.read_csv(path("data", "row", "cars_2025.csv"))

print("="*55)
print("  Raw Dataset Overview")
print("="*55)
print(f"  Shape      : {raw.shape[0]:,} rows  ×  {raw.shape[1]} columns")
print(f"  Columns    : {list(raw.columns)}")
print(f"  Memory     : {raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print()
print("Data types:")
print(raw.dtypes)

In [ ]:
# Missing value summary
missing = raw.isnull().sum()
missing_pct = (missing / len(raw) * 100).round(2)
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
missing_df = missing_df[missing_df["Missing Count"] > 0].sort_values("Missing %", ascending=False)
print("Columns with missing values:")
print(missing_df.to_string())

In [ ]:
# Sample rows
print("Sample rows from raw dataset:")
raw.head(5)

In [ ]:
# Price column inspection — it's stored as a string like "1,640,000 EGP"
print("Price column sample values:")
print(raw["price"].dropna().head(10).tolist())
print()
print(f"Rows with missing price: {raw['price'].isnull().sum():,}")

---
## 🧹 Stage 2 — Data Cleaning

Clean the raw data without modifying the original file.

**Cleaning steps:**
1. Parse and validate the `price` column → create `price_egp` (numeric EGP)
2. Exclude rows where price is missing (109 rows)
3. Remove exact duplicates (125 rows)
4. Investigate and correct suspicious `year` values
5. Parse `mileage` → `mileage_km` (numeric)
6. Recover `transmission` from the `features` column where possible

In [ ]:
# Load cleaned dataset (output of Stage 2 notebook)
df = pd.read_csv(path("data", "processed", "used_cars_cleaned.csv"))

print("="*55)
print("  Cleaned Dataset Summary")
print("="*55)
print(f"  Shape   : {df.shape[0]:,} rows  ×  {df.shape[1]} columns")
print(f"  Columns : {list(df.columns)}")
print()
print("Data types:")
print(df.dtypes)

In [ ]:
# Data cleaning pipeline summary
summary_data = {
    "Step": [
        "Initial raw rows",
        "Excluded — price missing",
        "Excluded — exact duplicates",
        "Final cleaned rows",
    ],
    "Count": [24688, 109, 125, 24454],
}
summary_df = pd.DataFrame(summary_data)
print("Data Cleaning Pipeline Summary:")
print(summary_df.to_string(index=False))

In [ ]:
# Missing values in cleaned dataset
missing_clean = df.isnull().sum()
missing_clean_pct = (missing_clean / len(df) * 100).round(2)
missing_clean_df = pd.DataFrame({"Missing Count": missing_clean, "Missing %": missing_clean_pct})
missing_clean_df = missing_clean_df[missing_clean_df["Missing Count"] > 0].sort_values("Missing %", ascending=False)
print("Remaining missing values in cleaned dataset:")
print(missing_clean_df.to_string())
print()
print("Note:")
print("  - mileage_km: 5.64% missing -> handled via hierarchical imputation in Stage 4")
print("  - transmission: 41.17% missing -> partial recovery via 'features' text in Stage 4")

---
## 📊 Stage 3 — Exploratory Data Analysis (EDA)

Understand distributions, relationships, and patterns in the data.

In [ ]:
# ── 3.1 Target Variable: Price Distribution ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw price
axes[0].hist(df["price_egp"].dropna() / 1e6, bins=60, color="#2196F3", edgecolor="white", linewidth=0.5)
axes[0].set_xlabel("Price (Million EGP)")
axes[0].set_ylabel("Count")
axes[0].set_title("Price Distribution (Raw)")
axes[0].axvline(df["price_egp"].median() / 1e6, color="red", linestyle="--", label=f"Median: {df['price_egp'].median()/1e6:.2f}M")
axes[0].legend()

# Log-transformed price
axes[1].hist(np.log1p(df["price_egp"].dropna()), bins=60, color="#4CAF50", edgecolor="white", linewidth=0.5)
axes[1].set_xlabel("log(1 + price_egp)")
axes[1].set_ylabel("Count")
axes[1].set_title("Price Distribution (Log-transformed)")

plt.suptitle("Price Distribution: Raw vs Log-transformed", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"Price statistics (EGP):")
print(df["price_egp"].describe().apply(lambda x: f"{x:,.0f}"))

In [ ]:
# ── 3.2 Year Distribution ───────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
year_counts = df["year"].dropna().astype(int).value_counts().sort_index()
ax.bar(year_counts.index, year_counts.values, color="#9C27B0", edgecolor="white", linewidth=0.4)
ax.set_xlabel("Model Year")
ax.set_ylabel("Number of Listings")
ax.set_title("Car Model Year Distribution")
ax.set_xticks(range(int(year_counts.index.min()), int(year_counts.index.max()) + 1, 2))
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

print(f"Year range: {df['year'].min():.0f} – {df['year'].max():.0f}")
print(f"Most common year: {df['year'].mode()[0]:.0f}")

In [ ]:
# ── 3.3 Price vs Year ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
df_sample = df[df["price_egp"] <= 5_000_000].dropna(subset=["year", "price_egp"])
medians = df_sample.groupby(df_sample["year"].astype(int))["price_egp"].median() / 1_000_000
ax.bar(medians.index, medians.values, color="#FF9800", edgecolor="white", linewidth=0.4)
ax.set_xlabel("Model Year")
ax.set_ylabel("Median Price (Million EGP)")
ax.set_title("Median Price by Model Year (price ≤ 5M EGP)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.4 Top 20 Car Companies ─────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
top20 = df["company"].value_counts().head(20)
ax.barh(top20.index[::-1], top20.values[::-1], color="#00BCD4")
ax.set_xlabel("Number of Listings")
ax.set_title("Top 20 Car Companies by Listing Count")
for i, v in enumerate(top20.values[::-1]):
    ax.text(v + 20, i, f"{v:,}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

print(f"Unique companies: {df['company'].nunique()}")
print(f"Unique models   : {df['model'].nunique()}")

In [ ]:
# ── 3.5 Price by Transmission ────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
trans_data = df[df["price_egp"] <= 5_000_000].dropna(subset=["transmission", "price_egp"])
trans_order = trans_data.groupby("transmission")["price_egp"].median().sort_values(ascending=False).index
sns.boxplot(data=trans_data, x="transmission", y="price_egp", order=trans_order,
            palette="Set2", ax=ax, showfliers=False)
ax.set_xlabel("Transmission")
ax.set_ylabel("Price (EGP)")
ax.set_title("Price by Transmission Type (price ≤ 5M EGP)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
plt.tight_layout()
plt.show()

print("Transmission value counts:")
print(df["transmission"].value_counts(dropna=False))

In [ ]:
# ── 3.6 Price vs Mileage ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
sample = df[
    (df["price_egp"] <= 5_000_000) &
    (df["mileage_km"] <= 400_000)
].dropna(subset=["mileage_km", "price_egp"])
ax.hexbin(sample["mileage_km"] / 1000, sample["price_egp"] / 1e6,
          gridsize=40, cmap="YlOrRd", mincnt=1)
ax.set_xlabel("Mileage (Thousands km)")
ax.set_ylabel("Price (Million EGP)")
ax.set_title("Price vs Mileage Density (price ≤ 5M EGP)")
plt.colorbar(ax.collections[0], ax=ax, label="Count")
plt.tight_layout()
plt.show()

corr = sample[["mileage_km", "price_egp"]].corr().iloc[0, 1]
print(f"Pearson correlation (mileage vs price): {corr:.3f}")

In [ ]:
# ── 3.7 Missing value heatmap ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 3))
miss_mask = df.isnull().astype(int)
miss_cols = miss_mask.columns[miss_mask.sum() > 0]
sns.heatmap(miss_mask[miss_cols].T, cmap="Blues", cbar=False, ax=ax, yticklabels=True)
ax.set_title("Missing Values Heatmap (blue = missing)")
ax.set_xlabel("Row index")
plt.tight_layout()
plt.show()

---
## 🔧 Stage 4 — Feature Engineering & Preprocessing

Transform raw columns into model-ready numeric features.

**Feature engineering steps:**
1. **Target:** `log_price = log1p(price_egp)` — removes skew
2. **Car age:** `car_age = 2025 − year`
3. **Hierarchical mileage imputation:** 3-tier lookup by (company+model+year) → (company+model) → (company) → global median
4. **Feature flags:** `has_ac`, `has_power_steering`, `has_remote_control` from text
5. **Frequency encoding:** company, model, location replaced by their listing frequency
6. **Target encoding:** smoothed mean-log-price per company/model/location
7. **One-hot encoding:** transmission (3 classes), color (25 classes)
8. **Listing age:** days since posted

In [ ]:
try:
    import category_encoders as ce
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "category_encoders", "-q"])
    import category_encoders as ce

from sklearn.model_selection import train_test_split

# ── Load cleaned dataset ─────────────────────────────────────
df = pd.read_csv(path("data", "processed", "used_cars_cleaned.csv"))
print(f"Loaded cleaned data: {df.shape}")

In [ ]:
# ── 4.1 Target variable ──────────────────────────────────────
df["log_price"] = np.log1p(df["price_egp"])
print(f"log_price range: {df['log_price'].min():.2f} – {df['log_price'].max():.2f}")

# ── 4.2 Car age ──────────────────────────────────────────────
REF_YEAR = 2025
df["car_age"] = (REF_YEAR - df["year"]).clip(lower=0)

# ── 4.3 Feature flags from text ──────────────────────────────
def parse_features(feat_text):
    s = str(feat_text).lower() if pd.notna(feat_text) else ""
    return {
        "has_ac":             int("air conditioner" in s or " ac " in s or s.startswith("ac")),
        "has_power_steering": int("power steering" in s),
        "has_remote_control": int("remote control" in s),
        "feature_count":      len([t for t in s.split(",") if t.strip()]),
    }

feat_parsed = df["features"].apply(parse_features).apply(pd.Series)
df = pd.concat([df, feat_parsed], axis=1)

# ── 4.4 Listing age ──────────────────────────────────────────
REF_DATE = pd.Timestamp("2025-08-31")
df["date_posted_dt"] = pd.to_datetime(df["date_posted"], errors="coerce")
df["listing_age_days"] = (REF_DATE - df["date_posted_dt"]).dt.days.clip(lower=0).fillna(0)

print("Feature engineering complete.")
print(df[["car_age", "has_ac", "has_power_steering", "has_remote_control",
          "listing_age_days", "log_price"]].describe().round(2))

In [ ]:
# ── 4.5 Train / Test split (80/20, stratified by year bucket) ─
df["year_bucket"] = pd.cut(df["year"], bins=[1990, 2005, 2015, 2020, 2023, 2026],
                            labels=False, include_lowest=True).fillna(0)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42,
                                      stratify=df["year_bucket"])
train_df = train_df.drop(columns=["year_bucket"])
test_df  = test_df.drop(columns=["year_bucket"])
df       = df.drop(columns=["year_bucket"])

print(f"Train: {len(train_df):,}  |  Test: {len(test_df):,}")

In [ ]:
# ── 4.6 Hierarchical mileage imputation ─────────────────────
# Build lookup tables on TRAIN ONLY (no data leakage)
l1 = (train_df.dropna(subset=["mileage_km"])
        .groupby(["company", "model", "year"])["mileage_km"].median().to_dict())
l2 = (train_df.dropna(subset=["mileage_km"])
        .groupby(["company", "model"])["mileage_km"].median().to_dict())
l3 = (train_df.dropna(subset=["mileage_km"])
        .groupby("company")["mileage_km"].median().to_dict())
global_mileage_median = train_df["mileage_km"].median()

def impute_mileage(row):
    if pd.notna(row["mileage_km"]):
        return row["mileage_km"], 0
    k1 = (row["company"], row["model"], row["year"])
    k2 = (row["company"], row["model"])
    if   k1 in l1: return l1[k1], 1
    elif k2 in l2: return l2[k2], 1
    elif row["company"] in l3: return l3[row["company"]], 1
    return global_mileage_median, 1

for split, dff in [("train", train_df), ("test", test_df)]:
    res = dff.apply(impute_mileage, axis=1)
    dff["mileage_km_imputed"] = [r[0] for r in res]
    dff["mileage_is_missing"] = [r[1] for r in res]

print(f"Global mileage median (train): {global_mileage_median:,.0f} km")
print(f"Mileage imputed in train: {train_df['mileage_is_missing'].sum():,}")
print(f"Mileage imputed in test : {test_df['mileage_is_missing'].sum():,}")

In [ ]:
# ── 4.7 Mileage per year ─────────────────────────────────────
for dff in [train_df, test_df]:
    dff["mileage_per_year"] = dff["mileage_km_imputed"] / dff["car_age"].clip(lower=1)

# ── 4.8 Transmission encoding ────────────────────────────────
def encode_transmission(t):
    t = str(t).strip().capitalize()
    if "Auto" in t: return "Automatic"
    if "Man"  in t: return "Manual"
    return "Unknown"

for dff in [train_df, test_df]:
    dff["trans_clean"]           = dff["transmission"].apply(encode_transmission)
    dff["transmission_is_missing"] = (dff["trans_clean"] == "Unknown").astype(int)
    dff["transmission_Automatic"]  = (dff["trans_clean"] == "Automatic").astype(float)
    dff["transmission_Manual"]     = (dff["trans_clean"] == "Manual").astype(float)
    dff["transmission_Unknown"]    = (dff["trans_clean"] == "Unknown").astype(float)

# ── 4.9 Frequency encoding ───────────────────────────────────
freq_maps = {}
for col in ["company", "model", "location"]:
    counts = train_df[col].value_counts(normalize=True)
    freq_maps[col] = counts.to_dict()
    for dff in [train_df, test_df]:
        dff[f"{col}_freq"] = dff[col].map(counts).fillna(0.0)

print("Frequency encoding done.")

In [ ]:
# ── 4.10 Target encoding (smoothed, no leakage) ──────────────
try:
    import category_encoders as ce
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "category_encoders", "-q"])
    import category_encoders as ce

te_cols = ["company", "model", "location"]
target_encoder = ce.TargetEncoder(cols=te_cols, smoothing=10, min_samples_leaf=5)
target_encoder.fit(train_df[te_cols], train_df["log_price"])

te_train = target_encoder.transform(train_df[te_cols])
te_test  = target_encoder.transform(test_df[te_cols])

for col in te_cols:
    train_df[f"{col}_te"] = te_train[col].values
    test_df[f"{col}_te"]  = te_test[col].values

print("Target encoding done. Global prior (log_price mean):",
      round(train_df["log_price"].mean(), 4))

In [ ]:
# ── 4.11 One-hot encoding for color ──────────────────────────
KNOWN_COLORS = sorted([
    "Beige", "Black", "Blue", "Bronze", "Brown", "Champagne", "Cyan",
    "Dark blue", "Dark green", "Dark grey", "Dark red", "Eggplant", "Gold",
    "Gray", "Green", "Mocha", "Olive", "Orange", "Other", "Petroleum",
    "Purple", "Red", "Silver", "White", "Yellow",
])

def encode_color(color_val):
    c = str(color_val).strip().title() if pd.notna(color_val) else "Other"
    if c not in KNOWN_COLORS:
        c = "Other"
    return {f"color_{kc}": (1.0 if kc == c else 0.0) for kc in KNOWN_COLORS}

for dff in [train_df, test_df]:
    color_enc = dff["color"].apply(encode_color).apply(pd.Series)
    for col in color_enc.columns:
        dff[col] = color_enc[col].values

print(f"Color one-hot columns: {len(KNOWN_COLORS)}")

In [ ]:
# ── 4.12 Assemble final feature matrices ─────────────────────
FEATURE_NAMES = (
    ["year", "car_age", "mileage_km_imputed", "mileage_per_year",
     "mileage_is_missing", "transmission_is_missing",
     "has_ac", "has_power_steering", "has_remote_control", "listing_age_days"]
    + ["company_freq", "model_freq", "location_freq"]
    + ["transmission_Automatic", "transmission_Manual", "transmission_Unknown"]
    + ["company_te", "model_te", "location_te"]
    + [f"color_{c}" for c in KNOWN_COLORS]
)

X_train = train_df[FEATURE_NAMES].copy()
X_test  = test_df[FEATURE_NAMES].copy()
y_train_log = train_df["log_price"].values
y_test_log  = test_df["log_price"].values
y_test_egp  = test_df["price_egp"].values

print(f"Feature matrix shape — Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Total features: {len(FEATURE_NAMES)}")
print(f"Feature names: {FEATURE_NAMES[:8]} ... [{len(FEATURE_NAMES)} total]")

---
## 🤖 Stage 5 — Baseline Model Training & Evaluation

Train and evaluate beginner-friendly regression models:
- **DummyRegressor** — always predicts the median (baseline reference)
- **LinearRegression** — simple linear model
- **DecisionTreeRegressor** — single tree, interpretable
- **RandomForestRegressor** — ensemble of trees (best expected performance)

All models trained on **log_price**, predictions converted back to EGP via `expm1()`.

In [ ]:
def eval_model(name, y_true_egp, y_pred_egp):
    """Compute MAE, RMSE, R² in EGP units."""
    mae  = mean_absolute_error(y_true_egp, y_pred_egp)
    rmse = np.sqrt(mean_squared_error(y_true_egp, y_pred_egp))
    r2   = r2_score(y_true_egp, y_pred_egp)
    return {"Model": name, "MAE (EGP)": mae, "RMSE (EGP)": rmse, "R²": r2}

baseline_results = []

# ── DummyRegressor ───────────────────────────────────────────
dummy = DummyRegressor(strategy="median")
dummy.fit(X_train, y_train_log)
dummy_pred_log = dummy.predict(X_test)
dummy_pred_egp = np.expm1(dummy_pred_log)
baseline_results.append(eval_model("DummyRegressor", y_test_egp, dummy_pred_egp))
print("DummyRegressor trained.")

# ── LinearRegression ─────────────────────────────────────────
lr = LinearRegression()
lr.fit(X_train, y_train_log)
lr_pred_egp = np.expm1(lr.predict(X_test))
baseline_results.append(eval_model("LinearRegression", y_test_egp, lr_pred_egp))
print("LinearRegression trained.")

# ── DecisionTreeRegressor ────────────────────────────────────
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train_log)
dt_pred_egp = np.expm1(dt.predict(X_test))
baseline_results.append(eval_model("DecisionTree (Default)", y_test_egp, dt_pred_egp))
print("DecisionTree trained.")

# ── RandomForestRegressor ────────────────────────────────────
print("Training RandomForest (100 estimators) — this may take ~1-2 minutes ...")
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train_log)
rf_pred_egp = np.expm1(rf.predict(X_test))
baseline_results.append(eval_model("RandomForest (100 trees)", y_test_egp, rf_pred_egp))
print("RandomForest trained.")

In [ ]:
# ── Display baseline results ──────────────────────────────────
results_df = pd.DataFrame(baseline_results)
results_df["MAE (EGP)"]  = results_df["MAE (EGP)"].round(0).astype(int)
results_df["RMSE (EGP)"] = results_df["RMSE (EGP)"].round(0).astype(int)
results_df["R²"]         = results_df["R²"].round(4)
print("Baseline Model Comparison (Full Test Set, n=4,891):")
print(results_df.to_string(index=False))

In [ ]:
# ── Visualize MAE comparison ─────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
colors  = ["#F44336", "#FF9800", "#2196F3", "#4CAF50"]
bars    = ax.barh(results_df["Model"], results_df["MAE (EGP)"] / 1e3, color=colors)
ax.set_xlabel("MAE (Thousand EGP)")
ax.set_title("Baseline Model Comparison — MAE on Test Set")
for bar, val in zip(bars, results_df["MAE (EGP)"]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
            f"{val:,.0f} EGP", va="center", fontsize=9)
ax.set_xlim(0, results_df["MAE (EGP)"].max() / 1e3 * 1.35)
plt.tight_layout()
plt.show()

---
## 🔬 Stage 6 — Model Improvement (Hyperparameter Tuning)

Improve the best baseline (Random Forest) by tuning hyperparameters:
- Increase `n_estimators` to 200 (more trees = lower variance)
- Keep `random_state=42` for reproducibility

We also tune the Decision Tree with `max_depth` to reduce overfitting.

In [ ]:
# ── Tuned Decision Tree ──────────────────────────────────────
print("Tuning Decision Tree (max_depth=10) ...")
dt_tuned = DecisionTreeRegressor(max_depth=10, min_samples_leaf=5, random_state=42)
dt_tuned.fit(X_train, y_train_log)
dt_tuned_pred_egp = np.expm1(dt_tuned.predict(X_test))
dt_tuned_result   = eval_model("DecisionTree (max_depth=10)", y_test_egp, dt_tuned_pred_egp)
print(f"  MAE: {dt_tuned_result['MAE (EGP)']:,.0f} EGP")

# ── Tuned Random Forest ──────────────────────────────────────
print("Tuning RandomForest (200 estimators) — this may take ~2-3 minutes ...")
rf_tuned = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_tuned.fit(X_train, y_train_log)
rf_tuned_pred_egp = np.expm1(rf_tuned.predict(X_test))
rf_tuned_result   = eval_model("RandomForest (200 trees, BEST)", y_test_egp, rf_tuned_pred_egp)
print(f"  MAE: {rf_tuned_result['MAE (EGP)']:,.0f} EGP")

In [ ]:
# ── Full comparison: baseline vs tuned ───────────────────────
all_results = (
    baseline_results
    + [dt_tuned_result, rf_tuned_result]
)
all_df = pd.DataFrame(all_results)
all_df["MAE (EGP)"]  = all_df["MAE (EGP)"].round(0).astype(int)
all_df["RMSE (EGP)"] = all_df["RMSE (EGP)"].round(0).astype(int)
all_df["R²"]         = all_df["R²"].round(4)
print("Full Model Comparison — Full Test Set (n=4,891):")
print(all_df.to_string(index=False))

In [ ]:
# ── Feature importance from tuned RF ─────────────────────────
importances = pd.Series(rf_tuned.feature_importances_, index=FEATURE_NAMES)
top20_feat  = importances.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20_feat.index[::-1], top20_feat.values[::-1], color="#1976D2")
ax.set_xlabel("Feature Importance (Gini)")
ax.set_title("Top 20 Feature Importances — Tuned Random Forest (200 Trees)")
plt.tight_layout()
plt.show()

print("Top 5 most important features:")
print(top20_feat.head(5).round(4))

---
## ✅ Stage 7 — Final Model Validation

Load the pre-trained production model and verify results independently.

> **Note:** We load the saved model from disk rather than relying on the
> in-memory `rf_tuned` object above, to simulate real-world inference.

In [ ]:
# ── Load pre-trained production pipeline ─────────────────────
PKL_PATH  = path("models", "final_prediction_pipeline.pkl")
JLIB_PATH = path("models", "final_prediction_pipeline.joblib")

if os.path.isfile(PKL_PATH):
    with open(PKL_PATH, "rb") as fh:
        bundle = pickle.load(fh)
    print(f"Loaded bundle from: {PKL_PATH}")
elif os.path.isfile(JLIB_PATH):
    bundle = joblib.load(JLIB_PATH)
    print(f"Loaded bundle from: {JLIB_PATH}")
else:
    print("Pre-trained model not found — using in-notebook rf_tuned instead.")
    bundle = None

if bundle is not None:
    prod_model = bundle["model"]
    print(f"Model: {bundle.get('model_name', type(prod_model).__name__)}")
    print(f"Version: {bundle.get('version', 'N/A')}")
    print(f"Features: {len(bundle['feature_names'])} columns")

In [ ]:
# ── Recalculate metrics from saved test predictions ──────────
PRED_PATH = path("results", "tuned_test_predictions.csv")

if os.path.isfile(PRED_PATH):
    saved_preds = pd.read_csv(PRED_PATH)
    print("Columns in saved predictions:", list(saved_preds.columns))

    # Find actual and predicted columns (flexible naming)
    actual_col = next((c for c in saved_preds.columns if "actual" in c.lower()), None)
    pred_col   = next((c for c in saved_preds.columns
                       if "pred" in c.lower() and "tuned_rf" in c.lower()), None)
    if pred_col is None:
        pred_col = next((c for c in saved_preds.columns if "pred" in c.lower()), None)

    if actual_col and pred_col:
        y_act  = saved_preds[actual_col].values
        y_pred = saved_preds[pred_col].values
        # Convert log predictions if stored as log
        if y_act.max() < 1000:  # likely log scale
            y_act  = np.expm1(y_act)
            y_pred = np.expm1(y_pred)

        mae_full  = mean_absolute_error(y_act, y_pred)
        rmse_full = np.sqrt(mean_squared_error(y_act, y_pred))
        r2_full   = r2_score(y_act, y_pred)

        mask_diag = y_act <= 5_000_000
        mae_diag  = mean_absolute_error(y_act[mask_diag], y_pred[mask_diag])
        rmse_diag = np.sqrt(mean_squared_error(y_act[mask_diag], y_pred[mask_diag]))
        r2_diag   = r2_score(y_act[mask_diag], y_pred[mask_diag])

        print()
        print("="*60)
        print("  FINAL VERIFIED METRICS — Tuned Random Forest (200 trees)")
        print("="*60)
        print(f"  Full Test Set (n={len(y_act):,})")
        print(f"    MAE  : {mae_full:>12,.0f} EGP")
        print(f"    RMSE : {rmse_full:>12,.0f} EGP")
        print(f"    R²   : {r2_full:>12.4f}")
        print()
        print(f"  Mainstream Market (price ≤ 5M EGP, n={mask_diag.sum():,})")
        print(f"    MAE  : {mae_diag:>12,.0f} EGP")
        print(f"    RMSE : {rmse_diag:>12,.0f} EGP")
        print(f"    R²   : {r2_diag:>12.4f}")
        print("="*60)
    else:
        print(f"Could not identify columns. Found: {list(saved_preds.columns)}")
else:
    print("Saved predictions not found — computing live from rf_tuned ...")
    y_act  = y_test_egp
    y_pred = rf_tuned_pred_egp
    mae_full  = mean_absolute_error(y_act, y_pred)
    rmse_full = np.sqrt(mean_squared_error(y_act, y_pred))
    r2_full   = r2_score(y_act, y_pred)
    mask_diag = y_act <= 5_000_000
    mae_diag  = mean_absolute_error(y_act[mask_diag], y_pred[mask_diag])
    rmse_diag = np.sqrt(mean_squared_error(y_act[mask_diag], y_pred[mask_diag]))
    r2_diag   = r2_score(y_act[mask_diag], y_pred[mask_diag])
    print(f"Full Test MAE  : {mae_full:,.0f} EGP")
    print(f"Diagnostic MAE : {mae_diag:,.0f} EGP  (R² = {r2_diag:.4f})")

In [ ]:
# ── Actual vs Predicted plot (Mainstream ≤ 5M EGP) ──────────
try:
    y_a = y_act[mask_diag] / 1e6
    y_p = y_pred[mask_diag] / 1e6
except NameError:
    y_a = y_test_egp[y_test_egp <= 5_000_000] / 1e6
    y_p = rf_tuned_pred_egp[y_test_egp <= 5_000_000] / 1e6

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
axes[0].scatter(y_a, y_p, alpha=0.35, s=10, color="#1565C0")
lim = max(y_a.max(), y_p.max()) * 1.05
axes[0].plot([0, lim], [0, lim], "r--", linewidth=1.5, label="Perfect prediction")
axes[0].set_xlabel("Actual Price (Million EGP)")
axes[0].set_ylabel("Predicted Price (Million EGP)")
axes[0].set_title("Actual vs Predicted — Mainstream Market")
axes[0].legend()

# Residuals
residuals = (y_p - y_a) * 1e6 / 1e3  # in thousand EGP
axes[1].hist(residuals, bins=60, color="#4CAF50", edgecolor="white", linewidth=0.4)
axes[1].axvline(0, color="red", linestyle="--", linewidth=1.5)
axes[1].axvline(residuals.mean(), color="orange", linestyle="--",
                label=f"Mean: {residuals.mean():,.1f}k EGP")
axes[1].set_xlabel("Prediction Error (Thousand EGP)")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual Distribution — Mainstream Market")
axes[1].legend()

plt.suptitle("Tuned Random Forest (200 Trees) — Validation Results", fontsize=14,
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 🚀 Stage 8 — Prediction Pipeline Demo

Demonstrate the end-to-end prediction pipeline on new example cars.

The `predict_price()` function handles:
- Missing mileage (hierarchical imputation)
- Missing transmission (defaults to Unknown)
- Unseen brands (target encoder falls back to global prior)
- Unknown colors (mapped to `Other`)

In [ ]:
# ── Load the predict_price function ──────────────────────────
# Option A: Use the standalone module
import importlib.util, sys

module_path = path("models", "predict_pipeline.py")
if os.path.isfile(module_path):
    spec = importlib.util.spec_from_file_location("predict_pipeline", module_path)
    pp   = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(pp)
    predict_price = pp.predict_price
    print("predict_price loaded from models/predict_pipeline.py")
else:
    print("predict_pipeline.py not found — defining inline function ...")
    # Inline fallback using the loaded bundle
    KNOWN_COLORS_LOCAL = sorted([
        "Beige","Black","Blue","Bronze","Brown","Champagne","Cyan",
        "Dark blue","Dark green","Dark grey","Dark red","Eggplant","Gold",
        "Gray","Green","Mocha","Olive","Orange","Other","Petroleum",
        "Purple","Red","Silver","White","Yellow",
    ])
    def predict_price(car_dict):
        prep = bundle["preprocessor_config"]
        model_obj = bundle["model"]
        feat_names = bundle["feature_names"]
        year = float(car_dict.get("year", prep["train_year_median"]))
        car_age = max(0.0, float(prep["ref_year"]) - year)
        company = str(car_dict.get("company", "Other")).strip()
        model_name = str(car_dict.get("model", "Other")).strip()
        loc = str(car_dict.get("location", "Other")).strip()
        m = car_dict.get("mileage_km", car_dict.get("mileage"))
        if m is None or (isinstance(m, float) and np.isnan(m)):
            m_missing = 1
            k1, k2 = (company, model_name, year), (company, model_name)
            if k1 in prep["l1_lookup"]: imp_m = prep["l1_lookup"][k1]
            elif k2 in prep["l2_lookup"]: imp_m = prep["l2_lookup"][k2]
            elif company in prep["l3_lookup"]: imp_m = prep["l3_lookup"][company]
            else: imp_m = prep["global_mileage_median"]
        else:
            m_missing, imp_m = 0, float(m)
        t_raw = str(car_dict.get("transmission", "")).strip().capitalize()
        trans = "Automatic" if "Auto" in t_raw else ("Manual" if "Man" in t_raw else "Unknown")
        t_missing = 1 if trans == "Unknown" else 0
        feat_str = str(car_dict.get("features", "")).lower()
        has_ac = 1 if "air conditioner" in feat_str else int(bool(car_dict.get("has_ac", 0)))
        has_ps = 1 if "power steering" in feat_str else int(bool(car_dict.get("has_power_steering", 0)))
        has_rc = 1 if "remote control" in feat_str else int(bool(car_dict.get("has_remote_control", 0)))
        row = {
            "year": year, "car_age": car_age,
            "mileage_km_imputed": float(imp_m),
            "mileage_per_year": float(imp_m) / max(car_age, 1.0),
            "mileage_is_missing": m_missing, "transmission_is_missing": t_missing,
            "has_ac": has_ac, "has_power_steering": has_ps, "has_remote_control": has_rc,
            "listing_age_days": float(car_dict.get("listing_age_days", 0.0)),
            "company_freq": float(prep["freq_maps"]["company"].get(company, 0.0)),
            "model_freq": float(prep["freq_maps"]["model"].get(model_name, 0.0)),
            "location_freq": float(prep["freq_maps"]["location"].get(loc, 0.0)),
            "transmission_Automatic": 1.0 if trans == "Automatic" else 0.0,
            "transmission_Manual": 1.0 if trans == "Manual" else 0.0,
            "transmission_Unknown": 1.0 if trans == "Unknown" else 0.0,
        }
        te_df = pd.DataFrame([{"company": company, "model": model_name, "location": loc}])
        te_out = prep["target_encoder"].transform(te_df)
        row["company_te"]  = float(te_out["company"].iloc[0])
        row["model_te"]    = float(te_out["model"].iloc[0])
        row["location_te"] = float(te_out["location"].iloc[0])
        c_assigned = str(car_dict.get("color", "Other")).strip().title()
        if c_assigned not in KNOWN_COLORS_LOCAL: c_assigned = "Other"
        for c in KNOWN_COLORS_LOCAL: row[f"color_{c}"] = 1.0 if c == c_assigned else 0.0
        X_in = pd.DataFrame([row])[feat_names]
        return int(round(np.expm1(model_obj.predict(X_in)[0]), -3))

In [ ]:
# ── Run prediction examples ───────────────────────────────────
test_cases = [
    {
        "description": "Toyota Corolla 2020 | White | Automatic | Cairo | 45,000 km",
        "car": {
            "company": "Toyota", "model": "Corolla", "year": 2020,
            "mileage_km": 45000, "transmission": "Automatic",
            "color": "White", "location": "Cairo",
            "features": "Air Conditioner, Power Steering",
        },
    },
    {
        "description": "Hyundai Elantra 2018 | Silver | MISSING mileage & transmission",
        "car": {
            "company": "Hyundai", "model": "Elantra", "year": 2018,
            "color": "Silver", "location": "Alexandria",
        },
    },
    {
        "description": "Fiat 128 1995 | Red | Manual | Giza | 180,000 km  (budget classic)",
        "car": {
            "company": "Fiat", "model": "128", "year": 1995,
            "mileage_km": 180000, "transmission": "Manual",
            "color": "Red", "location": "Giza",
        },
    },
    {
        "description": "Rivian R1T 2023 | Black | UNSEEN brand | Cairo | 8,000 km",
        "car": {
            "company": "Rivian", "model": "R1T", "year": 2023,
            "mileage_km": 8000, "transmission": "Automatic",
            "color": "Black", "location": "Cairo",
        },
    },
    {
        "description": "Mercedes-Benz C 180 2022 | Black | Automatic | Cairo",
        "car": {
            "company": "Mercedes", "model": "C 180", "year": 2022,
            "mileage_km": 22000, "transmission": "Automatic",
            "color": "Black", "location": "Cairo",
            "features": "Air Conditioner, Power Steering, Remote Control",
        },
    },
]

print("=" * 65)
print("  Egyptian Used Car Price Estimator — Example Predictions")
print("=" * 65)
for tc in test_cases:
    price = predict_price(tc["car"])
    print(f"  {tc['description']}")
    print(f"    Predicted Price: {price:>13,} EGP\n")

---
## 📈 Final Results Summary

### Model Selection Rationale

| Model | Full MAE | Mainstream MAE | Mainstream R² | Selected |
|-------|---------|----------------|--------------|---------|
| DummyRegressor (median) | ~909,002 EGP | ~427,485 EGP | −0.061 | No |
| LinearRegression | ~770,459 EGP | ~296,254 EGP | 0.409 | No |
| DecisionTree (default) | ~823,699 EGP | ~382,942 EGP | −446 | No |
| DecisionTree (depth=10) | ~601,821 EGP | ~160,859 EGP | 0.479 | No |
| RandomForest (100 trees) | ~566,399 EGP | ~121,477 EGP | 0.829 | No |
| **RandomForest (200 trees)** | **~565,695 EGP** | **~121,259 EGP** | **0.825** | **✅ Yes** |

### Key Findings

1. **Log-price transformation** dramatically improves model performance over raw-price training
2. **Random Forest** outperforms all other beginner-friendly models
3. The model explains **82.5% of price variance** in the mainstream market (≤ 5M EGP)
4. **Target encoding** and **hierarchical mileage imputation** are the most impactful engineering decisions

### Model Limitations

- **High-value outliers:** Ultra-rare cars (> 10M EGP) cause high full-test RMSE; the model is optimized for the mainstream market
- **Temporal scope:** Data reflects Egyptian used car market as of August 2025
- **Condition data absent:** No mechanical condition or accident history in listings

---
## 📦 Project Deliverables

All project files are in the `ORG_project/` folder:

```
ORG_project/
├── data/
│   ├── row/cars_2025.csv                     # Original raw dataset (READ-ONLY)
│   └── processed/
│       ├── used_cars_cleaned.csv              # Stage 2 output (24,454 × 14)
│       └── used_cars_engineered.csv           # Stage 4 output
├── models/
│   ├── final_prediction_pipeline.joblib       # Production bundle (.joblib)
│   ├── final_prediction_pipeline.pkl          # Production bundle (.pkl for Web App)
│   └── predict_pipeline.py                   # Standalone inference module
├── notebooks/
│   ├── 01_dataset_inspection.ipynb
│   ├── 02_data_cleaning.ipynb
│   ├── 03_exploratory_data_analysis.ipynb
│   ├── 04_feature_engineering_preprocessing.ipynb
│   ├── 05_baseline_model_training_evaluation.ipynb
│   ├── 06_model_improvement.ipynb
│   ├── 07_final_model_validation.ipynb
│   └── Egyptian_Used_Cars_Final.ipynb         # This notebook (complete project)
├── reports/
│   ├── stage_5_1_results_validation.md
│   ├── stage_6_model_improvement.md
│   ├── stage_7_final_validation.md
│   └── final_project_assembly.md
├── results/
│   ├── model_comparison_results_validated.csv
│   └── tuned_test_predictions.csv
└── requirements.txt
```

In [ ]:
# Final project artifact inventory
import os

artifacts = {
    "Raw Dataset":          path("data", "row", "cars_2025.csv"),
    "Cleaned Dataset":      path("data", "processed", "used_cars_cleaned.csv"),
    "Pipeline (.joblib)":   path("models", "final_prediction_pipeline.joblib"),
    "Pipeline (.pkl)":      path("models", "final_prediction_pipeline.pkl"),
    "Inference Module":     path("models", "predict_pipeline.py"),
    "Final Notebook":       path("notebooks", "Egyptian_Used_Cars_Final.ipynb"),
    "Requirements":         path("requirements.txt"),
    "Final Report":         path("reports", "final_project_assembly.md"),
}

print("Project Artifact Inventory:")
print("-" * 60)
for name, fpath in artifacts.items():
    exists = os.path.isfile(fpath)
    size_mb = os.path.getsize(fpath) / 1e6 if exists else 0
    status = f"{size_mb:>7.1f} MB" if exists else "  MISSING"
    print(f"  {'OK' if exists else 'XX'}  {name:<25}  {status}")
print("-" * 60)
print("All checks complete.")